# Лабораторная работа № 3 — ЭТАЛОННОЕ РЕШЕНИЕ

## Модель камеры, проективная геометрия и стереопара

**Дисциплина:** Введение в технологии виртуальной дополненной реальности и робототехники
**Тема курса:** 4. Проективная геометрия и модель камеры. Стереоскопия и рендеринг в VR

---

**Для преподавателя.** Не выдавать студентам до сдачи работы.

---

### Цель работы

Реализовать полный конвейер «модель — мир — камера — изображение», смоделировать
стереопару и убедиться численно, что погрешность восстановления глубины растёт
пропорционально квадрату расстояния и обратно пропорционально базе.

### Что понадобится

`numpy`, `matplotlib`. Графические библиотеки не нужны: рендеринг вы напишете сами.
`opencv-python` — только для сверки в заданиях 3 и 7.

### Правило практикума

Алгоритм вы реализуете сами. `cv2.projectPoints`, `cv2.undistortPoints`,
`cv2.triangulatePoints` допустимо использовать **только** для сверки — и такую сверку
нужно показать явно.

## Теоретическая справка

### Модель камеры-обскуры

Точка $X$ в системе координат камеры проецируется на плоскость изображения делением
на глубину. В однородных координатах это записывается линейно:

$$
\tilde{x} \;=\; K\,[\,R \mid t\,]\,\tilde{X},
\qquad
K = \begin{pmatrix} f & 0 & c_x \\ 0 & f & c_y \\ 0 & 0 & 1\end{pmatrix}.
$$

Матрица $K$ содержит **внутренние** параметры (фокусное расстояние в пикселях
и главную точку), пара $[R \mid t]$ — **внешние**, то есть положение камеры в мире.
Именно здесь пригождается аппарат тем 2 и 3: $[R \mid t]$ — это в точности элемент
$SE(3)$, только записанный блоком $3\times4$.

Заметьте, что нижняя строка полной матрицы $4\times4$ здесь перестала быть $(0,0,0,1)$:
преобразование стало **проективным**, а не аффинным. Отсюда деление на третью
координату — то самое, из-за которого далёкие предметы кажутся меньше.

### Дисторсия

Реальный объектив искажает изображение. Стандартная модель Брауна — Конради
в нормированных координатах ($x_n = (u - c_x)/f$):

$$
\begin{aligned}
x_d &= x\,(1 + k_1 r^2 + k_2 r^4) + 2p_1 x y + p_2 (r^2 + 2x^2),\\
y_d &= y\,(1 + k_1 r^2 + k_2 r^4) + p_1 (r^2 + 2y^2) + 2p_2 x y,
\end{aligned}
\qquad r^2 = x^2 + y^2 .
$$

Первые слагаемые — радиальная дисторсия (бочка при $k_1 < 0$, подушка при $k_1 > 0$),
вторые — тангенциальная, от перекоса матрицы относительно оптической оси.
Обратного преобразования в замкнутой форме нет: его находят итерациями.

Ровно эта же модель применяется в VR-шлеме, только в обратную сторону: изображение
заранее искажают, чтобы линза выпрямила его обратно.

### Стереопара

Две камеры с одинаковой ориентацией, разнесённые на базу $B$ вдоль оси $x$.
Точка с глубиной $z$ проецируется в них со сдвигом — **диспаратностью**:

$$d = u_L - u_R = \frac{fB}{z}, \qquad\text{откуда}\qquad z = \frac{fB}{d}.$$

Дифференцируя, получаем закон роста погрешности:

$$\delta z = \frac{z^2}{fB}\,\delta d .$$

Это соотношение вы проверите численно. Из него следуют все инженерные решения:
почему стереозрение не работает вдаль, почему у марсоходов камеры разнесены
на десятки сантиметров и почему в VR-шлеме межзрачковое расстояние — параметр модели,
а не настройка удобства.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## Задание 1. Модель камеры и матрица проекции

Ожидаемо: пиксель ровно $(640{,}000;\ 360{,}000)$, $\det R = 1$, $\|R^{\mathsf T}R - I\| \sim 3\cdot10^{-16}$.

In [2]:
def intrinsics(f, cx, cy):
    return np.array([[f, 0.0, cx], [0.0, f, cy], [0.0, 0.0, 1.0]])

def look_at(eye, target, up=(0.0, 0.0, 1.0)):
    """Возвращает (R, t): X_cam = R @ X_world + t. Оптическая ось камеры — +z."""
    eye, target, up = map(lambda v: np.asarray(v, float), (eye, target, up))
    z = target - eye; z /= np.linalg.norm(z)
    x = np.cross(up, z); x /= np.linalg.norm(x)
    y = np.cross(z, x)
    R = np.vstack([x, y, z])
    return R, -R @ eye

def project(K, R, t, X):
    """X: (N,3) в мировых координатах. Возвращает пиксели (N,2) и глубину z."""
    X = np.atleast_2d(np.asarray(X, float))
    Xc = X @ R.T + t
    uvw = Xc @ K.T
    return uvw[:, :2] / uvw[:, 2:3], Xc[:, 2]

K = intrinsics(f=800.0, cx=640.0, cy=360.0)
R0, t0 = look_at([0, -6, 2], [0, 0, 0.5])
uv, z = project(K, R0, t0, [[0.0, 0.0, 0.5]])
print('Задание 1: точка на оптической оси -> пиксель', np.round(uv[0], 4),
      ' главная точка (640, 360)')
assert np.allclose(uv[0], [640.0, 360.0], atol=1e-9)
assert abs(np.linalg.det(R0) - 1) < 1e-12
print('           det R = %.12f, ||RᵀR − I|| = %.2e'
      % (np.linalg.det(R0), np.linalg.norm(R0.T @ R0 - np.eye(3))))

Задание 1: точка на оптической оси -> пиксель [640. 360.]  главная точка (640, 360)
           det R = 1.000000000000, ||RᵀR − I|| = 3.14e-16


## Задание 2. Каркасный рендеринг без графических библиотек

**Ответ на вопрос:** при $z < 0$ деление на глубину даёт формально корректный, но геометрически бессмысленный результат — точка «отражается» и появляется на изображении с противоположной стороны. Проективное преобразование не различает точку и её антипод, поэтому отсечение по $z > 0$ приходится делать вручную. В графическом конвейере это называется near-plane clipping.

In [3]:
V = np.array([[x, y, zz] for x in (-1, 1) for y in (-1, 1) for zz in (0, 2)], float)
E = [(i, j) for i in range(8) for j in range(i + 1, 8)
     if np.sum(np.abs(V[i] - V[j]) > 1e-9) == 1]
uvV, zV = project(K, R0, t0, V)
print('Задание 2: рёбер у параллелепипеда %d (ожидается 12), все точки перед камерой: %s'
      % (len(E), bool(np.all(zV > 0))))
assert len(E) == 12 and np.all(zV > 0)

fig, ax = plt.subplots(figsize=(6.4, 3.6))
for i, j in E:
    ax.plot(*zip(uvV[i], uvV[j]), color='#1F2733', lw=1.4)
ax.plot(640, 360, '+', color='#E08A2E', ms=10)
ax.set_xlim(0, 1280); ax.set_ylim(720, 0); ax.set_aspect('equal')
ax.set_title('каркас в пикселях'); plt.tight_layout()
plt.savefig('/tmp/_wire.png', dpi=80); plt.close()

Задание 2: рёбер у параллелепипеда 12 (ожидается 12), все точки перед камерой: True


## Задание 3. Дисторсия и её компенсация

**Ответы на вопросы:** отрицательный $k_1$ даёт бочкообразную дисторсию (прямые выгибаются наружу), положительный — подушкообразную. Обратное преобразование полиномиально по $x_d$ не выражается: уравнение относительно $x$ имеет степень 5 и выше. Простая итерация сходится за 4—6 шагов; 30 взято с большим запасом — предложите студентам проверить это самим.

In [4]:
def distort(xn, k1, k2, p1, p2):
    x, y = xn[:, 0], xn[:, 1]
    r2 = x * x + y * y
    rad = 1 + k1 * r2 + k2 * r2 * r2
    xd = x * rad + 2 * p1 * x * y + p2 * (r2 + 2 * x * x)
    yd = y * rad + p1 * (r2 + 2 * y * y) + 2 * p2 * x * y
    return np.column_stack([xd, yd])

def undistort(xd, k1, k2, p1, p2, iters=30):
    x = xd.copy()
    for _ in range(iters):
        x = x - (distort(x, k1, k2, p1, p2) - xd)
    return x

k1, k2, p1, p2 = -0.28, 0.09, 1e-3, -8e-4
g = np.linspace(-0.55, 0.55, 21)
XN = np.column_stack([np.repeat(g, 21), np.tile(g, 21)])
XD = distort(XN, k1, k2, p1, p2)
XU = undistort(XD, k1, k2, p1, p2)
print('Задание 3: макс. сдвиг из-за дисторсии %.4f (в норм. координатах)'
      % np.abs(XD - XN).max())
print('           max ||undistort(distort(x)) − x|| = %.2e' % np.abs(XU - XN).max())
assert np.abs(XU - XN).max() < 1e-12

fig, axs = plt.subplots(1, 2, figsize=(9, 4.2))
for a, P, ttl in ((axs[0], XN, 'идеальная сетка'), (axs[1], XD, 'после дисторсии')):
    a.plot(P[:, 0], P[:, 1], '.', ms=3, color='#1F2733')
    a.set_aspect('equal'); a.set_title(ttl); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig('/tmp/_dist.png', dpi=80); plt.close()

Задание 3: макс. сдвиг из-за дисторсии 0.0767 (в норм. координатах)
           max ||undistort(distort(x)) − x|| = 1.33e-15


## Задание 4. Стереопара и восстановление глубины

**Ответы на вопросы:** при удвоении глубины диспаратность уменьшается вдвое — зависимость обратно пропорциональная. Бесконечно удалённой точке отвечает нулевая диспаратность: проекции в обеих камерах совпадают. Именно поэтому за некоторым расстоянием стереозрение перестаёт различать глубины.

In [5]:
f, B, cx = 800.0, 0.12, 640.0

def stereo_project(P, f, B, cx, cy=360.0):
    """P: (N,3) в системе левой камеры. Правая смещена на +B вдоль оси x."""
    X, Y, Z = P[:, 0], P[:, 1], P[:, 2]
    uL = f * X / Z + cx
    uR = f * (X - B) / Z + cx
    v = f * Y / Z + cy
    return uL, uR, v

Ztrue = np.array([0.5, 1.0, 2.0, 4.0, 8.0, 16.0])
P = np.column_stack([np.full_like(Ztrue, 0.1), np.zeros_like(Ztrue), Ztrue])
uL, uR, _ = stereo_project(P, f, B, cx)
d = uL - uR
Zest = f * B / d
print('Задание 4: z истинное  ', np.round(Ztrue, 3))
print('           диспаратность', np.round(d, 3))
print('           z по формуле ', np.round(Zest, 6))
print('           макс. расхождение %.2e' % np.abs(Zest - Ztrue).max())
assert np.abs(Zest - Ztrue).max() < 1e-10

Задание 4: z истинное   [ 0.5  1.   2.   4.   8.  16. ]
           диспаратность [192.  96.  48.  24.  12.   6.]
           z по формуле  [ 0.5  1.   2.   4.   8.  16. ]
           макс. расхождение 0.00e+00


## Задание 5. Погрешность глубины растёт как квадрат расстояния

**Ответы на вопросы:** совпадение количественное: при $z = 4$ м эксперимент даёт 0,0494 м, теория — 0,0497 м. Порог 10 % достигается там, где $z\sigma_d/(fB) = 0{,}1$, то есть при $z \approx 38$ м. Отклонение вверх на больших $z$ — следствие нелинейности: при малой диспаратности шум может приблизить её к нулю, и оценка $fB/d$ уходит в бесконечность. Среднее по такому распределению строго больше, чем даёт линеаризация.

In [6]:
sigma = 0.25          # шум диспаратности, пиксели
Zs = np.logspace(np.log10(0.5), np.log10(30), 18)
rms = []
for zt in Zs:
    dd = f * B / zt + rng.normal(0, sigma, 20000)
    dd = dd[dd > 1e-6]
    rms.append(np.sqrt(np.mean((f * B / dd - zt) ** 2)))
rms = np.array(rms)
slope = np.polyfit(np.log10(Zs[:12]), np.log10(rms[:12]), 1)[0]
pred = Zs ** 2 / (f * B) * sigma
print('Задание 5: наклон log-log = %.3f (теория 2)' % slope)
print('           численно/теория на z=4 м: %.4f / %.4f м'
      % (rms[np.argmin(abs(Zs - 4))], pred[np.argmin(abs(Zs - 4))]))
assert 1.8 < slope < 2.2

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.loglog(Zs, rms, 'o-', label='эксперимент')
ax.loglog(Zs, pred, '--', label=r'теория $z^2\sigma_d/(fB)$')
ax.set_xlabel('истинная глубина z, м'); ax.set_ylabel('СКО оценки z, м')
ax.legend(); ax.grid(True, which='both', alpha=.3)
plt.tight_layout(); plt.savefig('/tmp/_zerr.png', dpi=80); plt.close()

Задание 5: наклон log-log = 2.000 (теория 2)
           численно/теория на z=4 м: 0.0494 / 0.0497 м


## Задание 6. Влияние базы

**Ответ на вопрос:** погрешность обратно пропорциональна базе, поэтому для двукратного улучшения базу надо удвоить. Ограничения: габариты устройства, уменьшение области перекрытия полей зрения, рост числа точек, видимых лишь одной камерой, и усложнение поиска соответствий из-за большего ракурсного различия.

In [7]:
Bs = np.linspace(0.03, 0.5, 24)
z_fix = 6.0
err_B = []
for b in Bs:
    dd = f * b / z_fix + rng.normal(0, sigma, 20000)
    err_B.append(np.sqrt(np.mean((f * b / dd - z_fix) ** 2)))
err_B = np.array(err_B)
sl_B = np.polyfit(np.log10(Bs), np.log10(err_B), 1)[0]
print('Задание 6: наклон ошибки по базе = %.3f (теория −1)' % sl_B)
assert -1.2 < sl_B < -0.8

Задание 6: наклон ошибки по базе = -1.006 (теория −1)


## Задание 7. Триангуляция в общем случае (дополнительное)

Ожидаемо: без шума максимум ошибки $\sim3\cdot10^{-15}$ м; при шуме 0,5 пикселя СКО около 4 мм. Обратите внимание: это та же схема «однородная система → SVD → правый сингулярный вектор при наименьшем сингулярном числе», что и в оценке гомографии (тема 5), и в графовом SLAM (тема 14).

In [8]:
def triangulate_dlt(P1, P2, x1, x2):
    A = np.vstack([x1[0] * P1[2] - P1[0], x1[1] * P1[2] - P1[1],
                   x2[0] * P2[2] - P2[0], x2[1] * P2[2] - P2[1]])
    X = np.linalg.svd(A)[2][-1]
    return X[:3] / X[3]

R1, t1 = look_at([0, -5, 1.5], [0, 0, 0.6])
R2, t2 = look_at([3.5, -3.5, 1.8], [0, 0, 0.6])
P1 = K @ np.hstack([R1, t1[:, None]])
P2 = K @ np.hstack([R2, t2[:, None]])
Xw = rng.uniform([-1, -1, 0], [1, 1, 2], (200, 3))
x1 = project(K, R1, t1, Xw)[0]
x2 = project(K, R2, t2, Xw)[0]
rec = np.array([triangulate_dlt(P1, P2, x1[i], x2[i]) for i in range(len(Xw))])
print('Задание 7: макс. ошибка триангуляции без шума %.2e м'
      % np.abs(rec - Xw).max())
assert np.abs(rec - Xw).max() < 1e-8
x1n = x1 + rng.normal(0, 0.5, x1.shape)
x2n = x2 + rng.normal(0, 0.5, x2.shape)
recn = np.array([triangulate_dlt(P1, P2, x1n[i], x2n[i]) for i in range(len(Xw))])
print('           при шуме 0,5 пикс: СКО %.4f м, максимум %.4f м'
      % (np.sqrt(np.mean((recn - Xw) ** 2)), np.abs(recn - Xw).max()))
print('\nВСЕ ПРОВЕРКИ ПРОЙДЕНЫ')

Задание 7: макс. ошибка триангуляции без шума 3.00e-15 м
           при шуме 0,5 пикс: СКО 0.0041 м, максимум 0.0227 м

ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ


## Критерии приёма работы

| Что проверяется | На что смотреть |
|---|---|
| Задание 1 | Точка на оси даёт ровно главную точку; проверены свойства $R$ |
| Задание 2 | Рёбра построены программно, а не выписаны руками; ось $v$ направлена вниз |
| Задание 3 | Есть обе сетки на одном рисунке и проверка обратимости |
| Задание 4 | Таблица трёх столбцов; расхождение на уровне машинной погрешности |
| Задание 5 | Экспериментальная и теоретическая кривые на одном графике; наклон посчитан |
| Задание 6 | Наклон близок к $-1$; назван практический предел увеличения базы |
| Выводы | Все вопросы разобраны с числами |
| Воспроизводимость | «Restart & Run All» отрабатывает без ошибок |

**Частые ошибки:**

1. Забыт переворот оси $v$ — картинка выходит зеркальной по вертикали.
2. В `look_at` перепутан знак $t$: правильно $t = -R\,\mathrm{eye}$, а не $+R\,\mathrm{eye}$.
   Признак — точка на оптической оси не попадает в главную точку.
3. Дисторсия применяется к пиксельным координатам вместо нормированных. Формально
   работает, но коэффициенты теряют смысл и не совпадают с калибровочными.
4. В задании 5 берут мало испытаний ($10^2$ вместо $2\cdot10^4$) — оценка СКО шумит
   и наклон получается далёким от 2.
5. Диспаратность считается как $u_R - u_L$ — знак минус, глубина отрицательная.

## Прикладной блок · Модель камеры в ROS 2 и Gazebo *(необязательный)*

Связка со стеком:

| Объект из работы | Контрагент в ROS 2 / Gazebo |
|---|---|
| `intrinsics(f, cx, cy)` → матрица K | поле `k` сообщения `sensor_msgs/CameraInfo`; класс `image_geometry::PinholeCameraModel` |
| коэффициенты `k1, k2, p1, p2` из `distort()` | поле `d` того же сообщения (`distortion_model: plumb_bob`) |
| `project(K, R, t, X)` — модель-мир-камера | репроекция в `image_pipeline`; привязка к миру через `frame_id` из /tf |
| формула `Z = f·B/d` из задания 4 | стерео-конвейер `stereo_image_proc` → `depth_image_proc` → облако точек |
| квадратичный рост ошибки глубины (задание 5) | причина, почему камеры с малой базой теряют дальность |

В Gazebo камера настраивается не матрицей, а полем зрения; связь — одна формула:
`f = width / (2·tan(fov/2))`. Соответствующий сенсор:

```xml
<sensor type="camera" name="front_cam">
  <camera>
    <horizontal_fov>0.6747</horizontal_fov>
    <image><width>1280</width><height>720</height></image>
  </camera>
</sensor>
```

Блок ниже работает на чистом NumPy, внешних библиотек не требует.

In [9]:
# Прикладной блок (необязательный): K и стерео в формате ROS 2 / Gazebo

# 1) матрица K из intrinsics() — это поле .k сообщения sensor_msgs/CameraInfo
camera_info = {
    'header': {'frame_id': 'camera'},
    'height': 720, 'width': 1280,
    'distortion_model': 'plumb_bob',
    'd': [k1, k2, p1, p2, 0.0],        # коэффициенты distort() из этой работы
    'k': [float(K[0, 0]), float(K[0, 1]), float(K[0, 2]),
          float(K[1, 0]), float(K[1, 1]), float(K[1, 2]),
          float(K[2, 0]), float(K[2, 1]), float(K[2, 2])],
    'r': np.eye(3).ravel().tolist(),   # rectification-матрица стереопары
    'p': [float(K[0, 0]), 0.0, float(K[0, 2]), 0.0,
          0.0, float(K[1, 1]), float(K[1, 2]), 0.0,
          0.0, 0.0, 1.0, 0.0],        # проекция; у правой камеры p[3] = -f·B
}
K_back = np.asarray(camera_info['k']).reshape(3, 3)
print('CameraInfo.k -> K: расхождение с intrinsics() = %.2e'
      % np.linalg.norm(K_back - K))
assert np.allclose(K_back, K, atol=1e-15)

# 2) Gazebo: из f и width получается horizontal_fov, и обратно — без потерь
w_img = 1280
fov_x = 2 * np.arctan(w_img / (2 * K[0, 0]))
f_back = w_img / (2 * np.tan(fov_x / 2))
print('f = %.1f пикс, width = %d -> horizontal_fov = %.4f рад = %.2f град'
      % (K[0, 0], w_img, fov_x, np.degrees(fov_x)))
assert abs(f_back - K[0, 0]) < 1e-9

# 3) стерео: конвейер depth_image_proc считает формулу задания 4
d_stand = f * B / Ztrue[0]
Z_back = f * B / d_stand
print('f·B/d: при f=%.0f, B=%.2f, d=%.3f -> Z = %.3f м (задание 4 без потерь)'
      % (f, B, d_stand, Z_back))
assert abs(Z_back - Ztrue[0]) < 1e-12
print('у правой камеры стереопары ROS поле P[3] = -f·B — та же база, что в stereo_project')
print()
print('ПРИКЛАДНОЙ БЛОК: ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ')

CameraInfo.k -> K: расхождение с intrinsics() = 0.00e+00
f = 800.0 пикс, width = 1280 -> horizontal_fov = 1.3495 рад = 77.32 град
f·B/d: при f=800, B=0.12, d=192.000 -> Z = 0.500 м (задание 4 без потерь)
у правой камеры стереопары ROS поле P[3] = -f·B — та же база, что в stereo_project

ПРИКЛАДНОЙ БЛОК: ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ
